# Epic 2 — Dataset preparation (US 2.1 / US 2.2)

From paired **images** + **void masks**, extract one crop per connected void region, optionally save **edge maps** for ControlNet-style conditioning.

**CLI (production):**
```bash
udm-epic2 extract-crops --image-dir data/real/train/images --mask-dir data/real/train/masks \
  -o data/epic2/crops -c configs/epic2_dataset.yaml
```

Outputs: `images/void/`, `masks/void/`, optional `edges/void/`, and `manifest.csv`. Next: `udm-epic2 train` / `generate` / `paste` (see README).

In [ ]:
import tempfile
from pathlib import Path

import cv2
import numpy as np

from udm_epic2.dataset.crops import CropConfig, process_crop_dataset

In [ ]:
# Demo: synthetic pair → crop folder + manifest
with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    imd = root / "im"
    msd = root / "ms"
    imd.mkdir()
    msd.mkdir()
    img = np.full((128, 128), 4000, dtype=np.uint16)
    m = np.zeros((128, 128), dtype=np.uint8)
    m[40:90, 40:90] = 255
    cv2.imwrite(str(imd / "chip.png"), img)
    cv2.imwrite(str(msd / "chip.png"), m)
    out = root / "epic2_out"
    cfg = CropConfig(min_component_area_px=100, padding_px=4, defect_class="void")
    man = process_crop_dataset(imd, msd, out, cfg, write_edges=True)
    print("manifest:", man.read_text()[:500])